# 05b · 学習出現曲の補助評価とA/B/C統合表

主結果・主スコア・学習モデルを変更しない追加分析。29日/149日の期限対応集合と、両方に出現した共通集合を全モデル共通で使う。生スコアAUCの前後差・95%区間・固定最低値率と明示的欠損率を記録。Aは既存の149日モデルのholdout NDCGを再利用し、T1/T2のB/Cとのτ-bを同じ表へまとめる。

In [1]:
from pathlib import Path
import hashlib
import polars as pl
from IPython.display import display
from relisten import config
from relisten.supplementary import run_seen_item_analysis,combined_rank_table,write_supplementary_report
primary_names=["all_model_results.csv","main_test_results.csv","raw_score_model_metrics.csv","paired_model_differences.csv","top_model_paired_differences.csv","BENCHMARK_REPORT.md","model_head_coefficients.csv"]
primary_before={name:hashlib.sha256((config.RESULTS/name).read_bytes()).hexdigest() for name in primary_names}
ranks=combined_rank_table(pl.read_csv(config.RESULTS/"all_model_results.csv"),pl.read_csv(config.RESULTS/"task_a_next_item.csv"))
display(ranks)

family,b_user_auc,c_user_auc,a_ndcg_at_10,task,lower_days,rank_models,tau_a_c,p_a_c,tau_b_c,p_b_c,a_rank,b_rank,c_rank
str,f64,f64,f64,str,i32,i32,f64,f64,f64,f64,f64,f64,f64
"""ACT-R""",0.60491,0.616118,0.202728,"""T1""",1,8,-0.142857,0.719544,0.642857,0.031151,6.0,1.0,1.0
"""ItemKNN""",0.567966,0.559807,0.36852,"""T1""",1,8,-0.142857,0.719544,0.642857,0.031151,2.0,3.0,2.0
"""AudioCosine""",0.554621,0.550597,0.051773,"""T1""",1,8,-0.142857,0.719544,0.642857,0.031151,8.0,4.0,3.0
"""SASRec""",0.601128,0.544801,0.322591,"""T1""",1,8,-0.142857,0.719544,0.642857,0.031151,3.0,2.0,4.0
"""DecayPop""",0.545825,0.536446,0.226809,"""T1""",1,8,-0.142857,0.719544,0.642857,0.031151,5.0,6.0,5.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ACT-R""",0.514973,0.51052,0.202728,"""T2""",1,8,-0.285714,0.39876,0.642857,0.031151,6.0,5.0,4.0
"""MostPop""",0.510113,0.507429,0.168978,"""T2""",1,8,-0.285714,0.39876,0.642857,0.031151,7.0,7.0,5.0
"""iALS""",0.517953,0.501084,0.402587,"""T2""",1,8,-0.285714,0.39876,0.642857,0.031151,1.0,4.0,6.0


In [2]:
seen=run_seen_item_analysis()
write_supplementary_report(seen,ranks)
assert all(hashlib.sha256((config.RESULTS/name).read_bytes()).hexdigest()==digest for name,digest in primary_before.items()), "Primary results changed"
display(seen.filter((pl.col("lower_days")==config.LOWER_BOUND_DAYS)&(pl.col("split")=="test")))

seen-item auxiliary evaluation completed: light_features.parquet


seen-item auxiliary evaluation completed: ItemKNN_features.parquet


seen-item auxiliary evaluation completed: iALS_features.parquet


seen-item auxiliary evaluation completed: BPR_features.parquet


seen-item auxiliary evaluation completed: SASRec_features.parquet


family,task,split,route,protocol,lower_days,model_end_day,selection,seeds,pairs_before,pairs_after,users_before,users_after,minimum_fraction_before,minimum_fraction_after,fallback_fraction_before,fallback_fraction_after,auc_users_before,auc_users_after,user_auc_before,before_ci_low,before_ci_high,user_auc_after,after_ci_low,after_ci_high,auc_change,common_auc_users,common_user_auc_change,common_change_ci_low,common_change_ci_high
str,str,str,str,str,i64,i64,str,i64,i64,i64,i64,i64,f64,f64,f64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64
"""ACT-R""","""T1""","""test""","""C""","""raw_score""",1,149,"""matching_snapshot""",1,982716,966852,4485,4465,0.000001,0.000001,null,null,3953,3932,0.616118,0.611034,0.621153,0.616515,0.611505,0.62163,0.000397,3932,0.000014,-0.001188,0.001252
"""ACT-R""","""T1""","""test""","""C""","""raw_score""",1,149,"""seen_by_both""",1,982716,802137,4485,4422,0.000001,0.0,null,null,3953,3865,0.616118,0.611034,0.621153,0.61634,0.610792,0.621424,0.000222,3865,-0.000908,-0.003032,0.001175
"""AudioCosine""","""T1""","""test""","""C""","""raw_score""",1,149,"""seen_by_both""",1,982716,802137,4485,4422,0.073271,0.071044,0.073271,0.071044,3953,3865,0.550597,0.545987,0.555191,0.543995,0.538988,0.54894,-0.006601,3865,-0.006913,-0.008906,-0.004886
"""AudioCosine""","""T1""","""test""","""C""","""raw_score""",1,149,"""matching_snapshot""",1,982716,966852,4485,4465,0.073271,0.073536,0.073271,0.073536,3953,3932,0.550597,0.545987,0.555191,0.549723,0.545066,0.554266,-0.000873,3932,-0.00118,-0.002133,-0.00018
"""BPR""","""T1""","""test""","""C""","""raw_score""",1,149,"""matching_snapshot""",3,982716,966852,4485,4465,0.031892,0.016008,0.031892,0.016008,3953,3932,0.496336,0.491818,0.500784,0.500208,0.495722,0.504587,0.003872,3932,0.003613,0.002436,0.004801
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""MostPop""","""T2""","""test""","""C""","""raw_score""",1,149,"""matching_snapshot""",1,967884,783315,4819,4783,0.201503,0.013357,null,null,4124,3948,0.507429,0.502166,0.512431,0.546171,0.540557,0.551366,0.038741,3948,0.026833,0.02311,0.030533
"""SASRec""","""T2""","""test""","""C""","""raw_score""",1,149,"""matching_snapshot""",1,967884,783315,4819,4783,0.203013,0.015222,0.203013,0.015222,4124,3948,0.531674,0.52688,0.536589,0.585779,0.580617,0.591154,0.054105,3948,0.04112,0.037122,0.044939
"""SASRec""","""T2""","""test""","""C""","""raw_score""",1,149,"""seen_by_both""",1,967884,608786,4819,4738,0.203013,0.004603,0.203013,0.004603,4124,3717,0.531674,0.52688,0.536589,0.587182,0.581232,0.593132,0.055508,3717,0.040858,0.036086,0.045756


出力は `results/SUPPLEMENTARY_COMPARISON_REPORT.md`、`seen_item_auc_comparison.csv`、`seen_item_auc_by_seed.csv`、`seen_item_protocol.json`、`abc_rank_summary.csv`。絞り込みでAUC算出可能なユーザーが変わるため、母集団差を含む前後差と、同じユーザーだけの対応付き差を区別する。